# data.ipynb

Layer data: pemuatan dataset mentah, augmentasi offline, pipeline input `tf.data`, dan setiap visualisasi matplotlib terkait data.

**Kebijakan preprocessing.** Setiap gambar dan mask diubah ukurannya ke 512x512 dengan resize biasa (INTER_AREA untuk gambar, INTER_NEAREST untuk mask). Fungsi `standardize_image` / `standardize_mask` yang sama digunakan saat train, eval, dan inferensi, sehingga tidak ada ketidakcocokan train/serve.

Fungsi pipeline ini sebelumnya berada di kode pelatihan, yang memaksa pengimpor menjalankan efek samping mixed-precision. Fungsi ini ditempatkan di sini agar evaluasi dan inferensi dapat menggunakannya kembali dengan bersih. Muat dengan `%run data.ipynb`.

In [1]:
import os
import numpy as np
import cv2
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import albumentations as A

# Ukuran spasial target yang digunakan di semua tahap downstream.
H = 512
W = 512

In [2]:
try:
    _DEMO
except NameError:
    _DEMO = True  # True saat notebook ini dijalankan sendiri; disetel False oleh pemanggil %run

### Helper

In [3]:
def create_dir(path):
    """Buat direktori (dan parent-nya) jika belum ada."""
    if not os.path.exists(path):
        os.makedirs(path)

### Pemuatan dataset mentah

Muat path gambar/mask dan bagi menjadi train/test sebagai daftar **berpasangan**, sehingga gambar dan mask-nya selalu dibagi bersama dan tidak pernah terpisah.

In [4]:
def load_data(path, split=0.1):
    """
    Muat path gambar/mask mentah dan bagi menjadi train/test sebagai daftar berpasangan.

    Pembagian dilakukan pada pasangan (gambar, mask) bersama-sama, sehingga
    train_x[i] selalu berkorespondensi dengan train_y[i]. Membagi X dan Y
    secara independen (perilaku sebelumnya) hanya kebetulan cocok karena
    random_state sama; ini akan rusak diam-diam jika salah satu daftar berubah.
    """
    X = sorted(glob(os.path.join(path, "images", "*.jpg")))
    Y = sorted(glob(os.path.join(path, "masks",  "*.png")))

    print(f"[DATA] Ditemukan {len(X)} gambar dan {len(Y)} mask")

    if len(X) == 0 or len(Y) == 0:
        print("[PERINGATAN] Tidak ada gambar atau mask yang ditemukan!")
        return ([], []), ([], [])

    if len(X) != len(Y):
        print(f"[PERINGATAN] Jumlah gambar/mask tidak cocok ({len(X)} vs {len(Y)}). "
              f"Pemasangan berdasarkan urutan terurut mungkin tidak andal.")

    split_size = max(1, int(len(X) * split))
    train_x, test_x, train_y, test_y = train_test_split(
        X, Y, test_size=split_size, random_state=42
    )
    return (train_x, train_y), (test_x, test_y)

### Augmentasi

Delapan varian per gambar pelatihan (yang pertama adalah original tanpa modifikasi). Transformasi spasial diterapkan secara identik pada gambar dan mask-nya; mask di-binarisasi ulang ke 0/255 setelah setiap transformasi. Ditulis untuk albumentations 2.x.

In [5]:
def _build_augmentations():
    """
    Mengembalikan daftar tuple (label, transformasi_albumentations).

    Kompatibel dengan albumentations 2.x (signature CoarseDropout terbaru,
    tanpa alpha_affine pada ElasticTransform). Grayscale ditangani secara
    manual dengan cv2 karena kita menginginkan gambar grey 3-kanal,
    bukan kanal yang dihapus.
    """
    return [
        ("Flip Horizontal",
            A.HorizontalFlip(p=1.0)),
        ("Grayscale",
            None),  # ditangani secara manual dengan cv2
        ("Channel Shuffle",
            A.ChannelShuffle(p=1.0)),
        ("Coarse Dropout",
            A.CoarseDropout(
                num_holes_range=(3, 10),
                hole_height_range=(16, 32),
                hole_width_range=(16, 32),
                p=1.0,
            )),
        ("Rotasi +/-45 derajat",
            A.Rotate(limit=45, p=1.0)),
        ("Elastic Transform",
            A.ElasticTransform(alpha=120, sigma=6, p=1.0)),
        ("Brightness/Contrast",
            A.RandomBrightnessContrast(
                brightness_limit=0.3, contrast_limit=0.3, p=1.0
            )),
    ]

In [6]:
def augment_data(images, masks, save_path, augment=True):
    """
    Menulis hingga 8 varian per gambar ke save_path/image dan save_path/mask.

    Varian 0    : original (selalu ditulis)
    Varian 1-7  : augmentasi (hanya ketika augment=True)

    Setiap output diubah ukurannya ke 512x512 dengan kebijakan resize bersama
    dan setiap mask di-binarisasi ulang ke 0/255 setelah transformasi.

    Binarisasi memakai `> 0` (foreground = piksel non-nol), bukan `> 127`.
    Mask mentah dataset ini disimpan sebagai 0/1, sehingga ambang 127 akan
    mengosongkan SETIAP mask (semua piksel jadi 0). `> 0` robust terhadap
    mask 0/1 maupun 0/255.
    """
    image_dir = os.path.join(save_path, "image")
    mask_dir  = os.path.join(save_path, "mask")
    create_dir(image_dir)
    create_dir(mask_dir)

    augmentations = _build_augmentations() if augment else []

    print(f"[AUG] Memproses {len(images)} gambar "
          f"({'8 varian masing-masing' if augment else '1 varian masing-masing'})...")

    for x_path, y_path in tqdm(zip(images, masks), total=len(images)):
        name = os.path.basename(x_path).split(".")[0]

        img  = cv2.imread(x_path, cv2.IMREAD_COLOR)
        mask = cv2.imread(y_path, cv2.IMREAD_GRAYSCALE)

        if img is None or mask is None:
            print(f"[PERINGATAN] Tidak dapat memuat {x_path} atau {y_path}")
            continue

        # albumentations bekerja pada mask 3-kanal dengan bersih; konversi kembali nanti.
        mask3 = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)

        variants_img  = [img]
        variants_mask = [mask3]

        if augment:
            for label, aug in augmentations:
                if label == "Grayscale":
                    gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                    aug_img  = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
                    aug_mask = mask3.copy()
                else:
                    result   = aug(image=img, mask=mask3)
                    aug_img  = result["image"]
                    aug_mask = result["mask"]
                variants_img.append(aug_img)
                variants_mask.append(aug_mask)

        for idx, (vi, vm) in enumerate(zip(variants_img, variants_mask)):
            out_img  = standardize_image(vi)
            out_mask = vm
            if out_mask.ndim == 3:
                out_mask = cv2.cvtColor(out_mask, cv2.COLOR_BGR2GRAY)
            out_mask = standardize_mask(out_mask)
            out_mask = (out_mask > 0).astype(np.uint8) * 255

            cv2.imwrite(os.path.join(image_dir, f"{name}_{idx}.png"), out_img)
            cv2.imwrite(os.path.join(mask_dir,  f"{name}_{idx}.png"), out_mask)

### Preprocessing dan pipeline `tf.data`

Resize bersama, decode per-sampel (`read_image` / `read_mask` / `tf_parse`), dan `tf_dataset` (shuffle level dataset, drop_remainder pada training, prefetch AUTOTUNE).

In [7]:
def standardize_image(img, size=(W, H)):
    """Ubah ukuran gambar BGR uint8 ke `size` (INTER_AREA). Tidak melakukan apa-apa jika sudah berukuran sesuai."""
    if (img.shape[1], img.shape[0]) != size:
        img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
    return img


def standardize_mask(mask, size=(W, H)):
    """Ubah ukuran mask grayscale ke `size` dengan nearest-neighbour (tetap biner)."""
    if (mask.shape[1], mask.shape[0]) != size:
        mask = cv2.resize(mask, size, interpolation=cv2.INTER_NEAREST)
    return mask


def load_processed_data(path):
    """Glob pasangan PNG gambar/mask augmentasi di bawah path/image dan path/mask."""
    x = sorted(glob(os.path.join(path, "image", "*.png")))
    y = sorted(glob(os.path.join(path, "mask",  "*.png")))
    return x, y

In [8]:
def read_image(path):
    """Baca gambar BGR, ubah ukuran ke 512x512, skalakan ke [0,1] float32."""
    if isinstance(path, bytes):
        path = path.decode()
    x = cv2.imread(path, cv2.IMREAD_COLOR)
    x = standardize_image(x)
    x = x.astype(np.float32) / 255.0
    return x


def read_mask(path):
    """Baca mask, ubah ukuran ke 512x512, kembalikan biner float32 (0.0/1.0), HxWx1."""
    if isinstance(path, bytes):
        path = path.decode()
    x = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    x = standardize_mask(x)
    x = (x > 0).astype(np.float32)     # foreground = piksel non-nol; robust terhadap mask 0/1 atau 0/255
    x = np.expand_dims(x, axis=-1)
    return x


def tf_parse(x, y):
    """Bungkus read_image / read_mask untuk tf.data dan kembalikan bentuk statis."""
    import tensorflow as tf

    def _parse(xp, yp):
        return read_image(xp), read_mask(yp)

    x, y = tf.numpy_function(_parse, [x, y], [tf.float32, tf.float32])
    x.set_shape([H, W, 3])
    y.set_shape([H, W, 1])
    return x, y


def tf_dataset(X, Y, batch=2, shuffle=True, training=True):
    """
    Bangun pipeline tf.data.

    training=True membuang batch parsial terakhir (drop_remainder), yang menjaga
    ukuran batch konstan untuk BatchNormalization dan loss scaler. Validasi
    menyimpan setiap sampel. Shuffling terjadi pada level dataset dengan
    reshuffle_each_iteration, sehingga urutan berubah setiap epoch.
    """
    import tensorflow as tf

    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(X), 1000), seed=42,
                        reshuffle_each_iteration=True)
    ds = ds.map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch, drop_remainder=training)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

### Visualisasi: eksplorasi data

Resolusi dan aspek, overlay sampel, rasio piksel foreground, dan intensitas per kanal. Masing-masing menulis figur ke `plots/`.

In [9]:
def _mpl():
    """Import matplotlib lazy dengan backend non-interaktif."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    return plt

In [10]:
def visualize_data_exploration(data_path, save_dir="plots", max_samples=400):
    """
    Empat panel ikhtisar dataset mentah, disimpan ke plots/02a_data_exploration.png:
        - scatter resolusi (lebar vs tinggi)
        - histogram rasio aspek
        - distribusi ukuran file gambar terpilih
        - panel ringkasan teks (jumlah, rentang resolusi, aspek rata-rata)

    Hanya `max_samples` gambar pertama yang diukur untuk panel resolusi/aspek
    agar pemindaian tetap cepat pada dataset besar.
    """
    plt = _mpl()
    create_dir(save_dir)

    images = sorted(glob(os.path.join(data_path, "images", "*.jpg")))
    masks  = sorted(glob(os.path.join(data_path, "masks",  "*.png")))
    if not images:
        print("[PERINGATAN] Tidak ada gambar yang ditemukan untuk eksplorasi data.")
        return

    sample = images[:max_samples]
    widths, heights, sizes_kb = [], [], []
    for p in sample:
        img = cv2.imread(p, cv2.IMREAD_COLOR)
        if img is None:
            continue
        h, w = img.shape[:2]
        widths.append(w)
        heights.append(h)
        sizes_kb.append(os.path.getsize(p) / 1024.0)

    widths  = np.array(widths)
    heights = np.array(heights)
    aspects = widths / np.maximum(heights, 1)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].scatter(widths, heights, s=12, alpha=0.5, color="#2196F3", edgecolor="none")
    axes[0, 0].set_xlabel("Lebar (px)")
    axes[0, 0].set_ylabel("Tinggi (px)")
    axes[0, 0].set_title("Distribusi Resolusi Gambar", fontweight="bold")
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].hist(aspects, bins=30, color="#4CAF50", edgecolor="white")
    axes[0, 1].axvline(1.0, color="gray", linestyle="--", linewidth=1, label="Persegi (1:1)")
    axes[0, 1].set_xlabel("Rasio aspek (L / T)")
    axes[0, 1].set_ylabel("Jumlah")
    axes[0, 1].set_title("Histogram Rasio Aspek", fontweight="bold")
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    axes[1, 0].hist(sizes_kb, bins=30, color="#FF9800", edgecolor="white")
    axes[1, 0].set_xlabel("Ukuran file (KB)")
    axes[1, 0].set_ylabel("Jumlah")
    axes[1, 0].set_title("Distribusi Ukuran File Gambar Terpilih", fontweight="bold")
    axes[1, 0].grid(alpha=0.3)

    axes[1, 1].axis("off")
    summary = (
        f"Ringkasan dataset\n"
        f"----------------------------\n"
        f"Gambar (total)   : {len(images)}\n"
        f"Mask   (total)   : {len(masks)}\n"
        f"Sampel diukur    : {len(widths)}\n\n"
        f"Rentang lebar    : {int(widths.min())} - {int(widths.max())} px\n"
        f"Rentang tinggi   : {int(heights.min())} - {int(heights.max())} px\n"
        f"Aspek rata-rata  : {aspects.mean():.3f}\n"
        f"Ukuran target    : {W} x {H} (resize untuk pelatihan)\n"
    )
    axes[1, 1].text(0.02, 0.98, summary, va="top", ha="left", family="monospace",
                    fontsize=12, transform=axes[1, 1].transAxes)
    axes[1, 1].set_title("Ringkasan", fontweight="bold")

    fig.suptitle("Eksplorasi Dataset Mentah", fontsize=16, fontweight="bold")
    plt.tight_layout()
    out = os.path.join(save_dir, "02a_data_exploration.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Eksplorasi data disimpan: {out}")

In [11]:
def visualize_samples(data_path, save_dir="plots", n_samples=5):
    """
    Grid sampel mentah: original | mask | overlay, disimpan ke
    plots/02b_sample_data.png. Overlay mewarnai foreground merah sehingga
    kualitas anotasi mudah dinilai.
    """
    plt = _mpl()
    create_dir(save_dir)

    images = sorted(glob(os.path.join(data_path, "images", "*.jpg")))
    masks  = sorted(glob(os.path.join(data_path, "masks",  "*.png")))
    if not images or not masks:
        print("[PERINGATAN] Tidak ada sampel yang ditemukan untuk visualisasi sampel.")
        return

    n = min(n_samples, len(images))
    idxs = np.linspace(0, len(images) - 1, n).astype(int)

    fig, axes = plt.subplots(n, 3, figsize=(11, n * 3.4))
    if n == 1:
        axes = axes[np.newaxis, :]

    for col, title in enumerate(["Original", "Mask", "Overlay"]):
        axes[0, col].set_title(title, fontsize=12, fontweight="bold", pad=8)

    for row, i in enumerate(idxs):
        img  = cv2.imread(images[i], cv2.IMREAD_COLOR)
        mask = cv2.imread(masks[i],  cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            continue
        img  = standardize_image(img)
        mask = standardize_mask(mask)
        rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        overlay = rgb.copy()
        m = mask > 0
        overlay[m] = (0.55 * np.array([255, 60, 60]) + 0.45 * overlay[m]).astype(np.uint8)

        axes[row, 0].imshow(rgb)
        axes[row, 1].imshow(mask * 255, cmap="gray", vmin=0, vmax=255)
        axes[row, 2].imshow(overlay)
        for col in range(3):
            axes[row, col].axis("off")

    fig.suptitle("Sampel Gambar, Mask, dan Overlay", fontsize=15, fontweight="bold")
    plt.tight_layout()
    out = os.path.join(save_dir, "02b_sample_data.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Sampel data disimpan: {out}")

In [12]:
def visualize_foreground_distribution(data_path, save_dir="plots", max_samples=600):
    """
    Histogram rasio piksel manusia/foreground di seluruh mask, plus garis
    rata-rata. Disimpan ke plots/02c_foreground_ratio.png. Menunjukkan seberapa
    besar ketidakseimbangan kelas (sebagian besar mask adalah latar belakang).
    """
    plt = _mpl()
    create_dir(save_dir)

    masks = sorted(glob(os.path.join(data_path, "masks", "*.png")))
    if not masks:
        print("[PERINGATAN] Tidak ada mask yang ditemukan untuk distribusi foreground.")
        return

    sample = masks[:max_samples]
    ratios = []
    for p in sample:
        m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        ratios.append(float((m > 0).mean()))
    ratios = np.array(ratios)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.hist(ratios, bins=30, color="#9C27B0", edgecolor="white", alpha=0.85)
    ax.axvline(ratios.mean(), color="#F44336", linestyle="--", linewidth=2,
               label=f"Rata-rata = {ratios.mean():.3f}")
    ax.set_xlabel("Rasio piksel foreground (manusia)")
    ax.set_ylabel("Jumlah mask")
    ax.set_title(f"Distribusi Rasio Piksel Foreground (n={len(ratios)})",
                 fontsize=14, fontweight="bold")
    ax.legend()
    ax.grid(alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    out = os.path.join(save_dir, "02c_foreground_ratio.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Rasio foreground disimpan: {out}")

In [13]:
def visualize_channel_intensities(data_path, save_dir="plots", max_samples=150):
    """
    Histogram intensitas rata-rata per kanal (B, G, R) pada sampel gambar,
    disimpan ke plots/02d_channel_intensity.png. Berguna sebagai sanity check
    sebelum mengandalkan asumsi normalisasi ImageNet.
    """
    plt = _mpl()
    create_dir(save_dir)

    images = sorted(glob(os.path.join(data_path, "images", "*.jpg")))
    if not images:
        print("[PERINGATAN] Tidak ada gambar yang ditemukan untuk plot intensitas kanal.")
        return

    sample = images[:max_samples]
    hist = np.zeros((3, 256), dtype=np.float64)
    count = 0
    for p in sample:
        img = cv2.imread(p, cv2.IMREAD_COLOR)
        if img is None:
            continue
        for c in range(3):
            hist[c] += cv2.calcHist([img], [c], None, [256], [0, 256]).flatten()
        count += 1
    if count == 0:
        print("[PERINGATAN] Tidak ada gambar yang dapat dibaca untuk plot intensitas kanal.")
        return
    hist /= count

    fig, ax = plt.subplots(figsize=(11, 5.5))
    names  = ["Biru", "Hijau", "Merah"]
    colors = ["#2196F3", "#4CAF50", "#F44336"]
    x = np.arange(256)
    for c in range(3):
        ax.plot(x, hist[c], color=colors[c], linewidth=1.6, label=names[c])
        ax.fill_between(x, hist[c], color=colors[c], alpha=0.12)
    ax.set_xlabel("Intensitas piksel (0-255)")
    ax.set_ylabel("Rata-rata jumlah piksel per gambar")
    ax.set_title(f"Distribusi Intensitas Per Kanal (n={count})",
                 fontsize=14, fontweight="bold")
    ax.legend()
    ax.grid(alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    out = os.path.join(save_dir, "02d_channel_intensity.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Intensitas kanal disimpan: {out}")

### Visualisasi: augmentasi, pembagian, dan sanity check pipeline

In [14]:
def visualize_augmentations(img_path, mask_path, save_dir="plots"):
    """
    Grid 2x8: original plus 7 varian augmentasi. Baris atas gambar, baris
    bawah mask. Disimpan ke plots/03_augmentation_showcase.png.
    """
    plt = _mpl()
    create_dir(save_dir)

    img  = cv2.imread(img_path,  cv2.IMREAD_COLOR)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None:
        print("[PERINGATAN] Tidak dapat memuat sampel untuk visualisasi augmentasi.")
        return

    mask3 = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    augmentations = _build_augmentations()
    labels = ["Original"] + [lbl for lbl, _ in augmentations]
    imgs, masks_out = [img], [mask]

    for label, aug in augmentations:
        if label == "Grayscale":
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            ai   = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
            am   = mask3.copy()
        else:
            r  = aug(image=img, mask=mask3)
            ai = r["image"]
            am = r["mask"]
        imgs.append(ai)
        am_gray = cv2.cvtColor(am, cv2.COLOR_BGR2GRAY) if am.ndim == 3 else am
        masks_out.append(am_gray)

    n = len(labels)
    fig, axes = plt.subplots(2, n, figsize=(n * 2.8, 6))

    for col, (lbl, im, msk) in enumerate(zip(labels, imgs, masks_out)):
        axes[0, col].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        axes[0, col].set_title(lbl, fontsize=8, fontweight="bold")
        axes[0, col].axis("off")
        axes[1, col].imshow(msk, cmap="gray")
        axes[1, col].axis("off")

    fig.suptitle("Augmentasi Data - 8 Varian per Gambar Pelatihan",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    out = os.path.join(save_dir, "03_augmentation_showcase.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Showcase augmentasi disimpan: {out}")

In [15]:
def visualize_dataset_split(n_train_orig, n_test_orig, n_train_aug, save_dir="plots"):
    """
    Pie chart pembagian train/test di samping bar chart ukuran dataset sebelum
    dan sesudah augmentasi. Disimpan ke plots/04_dataset_split.png.
    """
    plt = _mpl()
    create_dir(save_dir)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    sizes  = [n_train_orig, n_test_orig]
    labels = [f"Train\n{n_train_orig} gambar\n(90%)",
              f"Test\n{n_test_orig} gambar\n(10%)"]
    colors = ["#2196F3", "#F44336"]
    axes[0].pie(sizes, labels=labels, colors=colors, autopct="%1.1f%%",
                startangle=90, textprops={"fontsize": 11})
    axes[0].set_title("Pembagian Train / Test", fontsize=13, fontweight="bold")

    categories = ["Set Pelatihan\nOriginal", "Set Pelatihan\nAugmentasi", "Set Test"]
    values     = [n_train_orig, n_train_aug, n_test_orig]
    bar_colors = ["#90CAF9", "#2196F3", "#F44336"]
    bars = axes[1].bar(categories, values, color=bar_colors, edgecolor="white",
                       linewidth=1.2, width=0.5)
    for bar, val in zip(bars, values):
        axes[1].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + max(values) * 0.01,
                     f"{val:,}", ha="center", va="bottom",
                     fontsize=11, fontweight="bold")
    axes[1].set_ylabel("Jumlah pasangan gambar-mask", fontsize=11)
    axes[1].set_title("Ukuran Dataset Sebelum vs Sesudah Augmentasi",
                      fontsize=13, fontweight="bold")
    axes[1].spines["top"].set_visible(False)
    axes[1].spines["right"].set_visible(False)
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    out = os.path.join(save_dir, "04_dataset_split.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Pembagian dataset disimpan: {out}")

In [16]:
def visualize_tf_batch(dataset, save_dir="plots", n_samples=4):
    """
    Ambil satu batch dari pipeline tf.data dan tampilkan pasangan gambar/mask
    yang didekode. Mengonfirmasi pipeline menghasilkan tensor dengan bentuk
    dan skala yang benar. Disimpan ke plots/04b_pipeline_batch.png.
    """
    plt = _mpl()
    create_dir(save_dir)

    batch = next(iter(dataset.take(1)))
    images, masks = batch
    images = images.numpy()
    masks  = masks.numpy()
    n = min(n_samples, images.shape[0])
    if n == 0:
        print("[PERINGATAN] Batch kosong; tidak dapat memvisualisasikan pipeline.")
        return

    fig, axes = plt.subplots(2, n, figsize=(n * 3, 6.2))
    if n == 1:
        axes = axes.reshape(2, 1)

    for col in range(n):
        rgb = (images[col][:, :, ::-1] * 255).clip(0, 255).astype(np.uint8)
        axes[0, col].imshow(rgb)
        axes[0, col].set_title(f"gambar[{col}]", fontsize=9)
        axes[0, col].axis("off")
        axes[1, col].imshow(masks[col][:, :, 0], cmap="gray")
        axes[1, col].set_title(f"mask[{col}]", fontsize=9)
        axes[1, col].axis("off")

    fig.suptitle(
        f"Batch tf.data yang Didekode  (gambar {images.shape[1:]}, "
        f"rentang [{images.min():.2f}, {images.max():.2f}])",
        fontsize=13, fontweight="bold")
    plt.tight_layout()
    out = os.path.join(save_dir, "04b_pipeline_batch.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Batch pipeline disimpan: {out}")

### Demo

Menjalankan tahap data lengkap (plot eksplorasi, showcase augmentasi, lalu menulis dataset augmentasi) ketika notebook ini dijalankan sendiri dan dataset mentah tersedia. Dilewati saat dimuat via `%run`.

In [17]:
if _DEMO and os.path.isdir(os.path.join('people_segmentation', 'images')):
    np.random.seed(42)
    create_dir("plots")

    data_path = "people_segmentation"
    (train_x, train_y), (test_x, test_y) = load_data(data_path)

    print(f"[PEMBAGIAN] Train : {len(train_x)} | Test : {len(test_x)}")

    if len(train_x) == 0 and len(test_x) == 0:
        print("[ERROR] Tidak ada data yang ditemukan. Jalankan segmentation_prep.ipynb terlebih dahulu.")
        raise SystemExit(1)

    # Plot eksplorasi pada dataset mentah.
    print("\n[VIZ] Membuat plot eksplorasi data...")
    visualize_data_exploration(data_path, save_dir="plots")
    visualize_samples(data_path, save_dir="plots")
    visualize_foreground_distribution(data_path, save_dir="plots")
    visualize_channel_intensities(data_path, save_dir="plots")

    if len(train_x) > 0:
        print("\n[VIZ] Membuat showcase augmentasi...")
        visualize_augmentations(train_x[0], train_y[0], save_dir="plots")

    n_train_aug = len(train_x) * 8
    visualize_dataset_split(len(train_x), len(test_x), n_train_aug, save_dir="plots")

    create_dir("new_data/train")
    create_dir("new_data/test")

    if len(train_x) > 0:
        print("\n[AUG] Memproses data pelatihan...")
        augment_data(train_x, train_y, "new_data/train", augment=True)

    if len(test_x) > 0:
        print("\n[AUG] Memproses data tes (tanpa augmentasi)...")
        augment_data(test_x, test_y, "new_data/test", augment=False)

    print("\n[SELESAI] Pemrosesan data selesai.")
    print(f"          Pasangan pelatihan : ~{len(train_x) * 8:,} (8 varian masing-masing)")
    print(f"          Pasangan tes       :  {len(test_x):,}")
    print("          Output: new_data/train/image|mask  dan  new_data/test/image|mask")
elif _DEMO:
    print("Dataset mentah tidak ditemukan di people_segmentation/. "
          "Jalankan segmentation_prep.ipynb terlebih dahulu, atau lewati jika sudah diaugmentasi.")

[DATA] Ditemukan 5678 gambar dan 5678 mask
[PEMBAGIAN] Train : 5111 | Test : 567

[VIZ] Membuat plot eksplorasi data...
[Plot] Eksplorasi data disimpan: plots\02a_data_exploration.png
[Plot] Sampel data disimpan: plots\02b_sample_data.png
[Plot] Rasio foreground disimpan: plots\02c_foreground_ratio.png
[Plot] Intensitas kanal disimpan: plots\02d_channel_intensity.png

[VIZ] Membuat showcase augmentasi...
[Plot] Showcase augmentasi disimpan: plots\03_augmentation_showcase.png
[Plot] Pembagian dataset disimpan: plots\04_dataset_split.png

[AUG] Memproses data pelatihan...
[AUG] Memproses 5111 gambar (8 varian masing-masing)...


100%|██████████| 5111/5111 [14:39<00:00,  5.81it/s]  



[AUG] Memproses data tes (tanpa augmentasi)...
[AUG] Memproses 567 gambar (1 varian masing-masing)...


100%|██████████| 567/567 [00:10<00:00, 52.95it/s]


[SELESAI] Pemrosesan data selesai.
          Pasangan pelatihan : ~40,888 (8 varian masing-masing)
          Pasangan tes       :  567
          Output: new_data/train/image|mask  dan  new_data/test/image|mask
